# Accuracy of IDF evacuation orders in Gaza

In [ ]:
!pip install deep-translator langchain langchain-google-genai


[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
Traceback (most recent call last):
  File "<frozen runpy>", line 189, in _run_module_as_main
  File "<frozen runpy>", line 148, in _get_module_details
  File "<frozen runpy>", line 112, in _get_module_details
  File "c:\Users\Gizem\OneDrive\School\Werk\volkskrant\gaza-evacuatieorders\venv\Lib\site-packages\spacy\__init__.py", line 10, in <module>
    from thinc.neural.util import prefer_gpu, require_gpu
  File "c:\Users\Gizem\OneDrive\School\Werk\volkskrant\gaza-evacuatieorders\venv\Lib\site-packages\thinc\neural\__init__.py", line 4, in <module>
    from ._classes.model import Model  # noqa: F401
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Gizem\OneDrive\School\Werk\volkskrant\gaza-evacuatieorders\venv\Lib\site-packages\thinc\neural\_classes\model.py", line 11, in <module>
    from ..train import Trainer
  File "c:\Users\Gizem\OneDrive\School\Werk\vol

5.5.2: Pulling from library/elasticsearch
219d2e45b4af: Pulling fs layer
f478c95dec23: Pulling fs layer
017932737cd4: Pulling fs layer
28b38dddf546: Pulling fs layer
980edaaff53b: Pulling fs layer
9a2de73c3385: Pulling fs layer
98d9d4b54ad0: Pulling fs layer
f822bfcfdea9: Pulling fs layer
b08144fb654d: Pulling fs layer
5f9ae4a86d71: Pulling fs layer
288ffe538f2f: Pulling fs layer
47b728c174e9: Pulling fs layer
14ed224fb73f: Pulling fs layer
a482fbcfe407: Pulling fs layer
92aff82bd83f: Pulling fs layer
f478c95dec23: Download complete
28b38dddf546: Download complete
98d9d4b54ad0: Download complete
47b728c174e9: Download complete
b08144fb654d: Download complete
9a2de73c3385: Download complete
288ffe538f2f: Download complete
017932737cd4: Download complete
5f9ae4a86d71: Download complete
14ed224fb73f: Download complete
980edaaff53b: Download complete
a482fbcfe407: Download complete
f822bfcfdea9: Download complete
219d2e45b4af: Download complete
219d2e45b4af: Pull complete
a482fbcfe407: Pul

'wget' is not recognized as an internal or external command,
operable program or batch file.
tar: Error opening archive: Failed to open 'geonames_index.tar.gz'
docker: Error response from daemon: create $(pwd)/geonames_index: "$(pwd)/geonames_index" includes invalid characters for a local volume name, only "[a-zA-Z0-9][a-zA-Z0-9_.-]" are allowed. If you intended to pass a host directory, use absolute path

Run 'docker run --help' for more information


In [ ]:
# Install dependencies
import os
import time
import asyncio
from dotenv import load_dotenv
import re
import pandas as pd
from deep_translator import GoogleTranslator
from datetime import datetime, timedelta
from langchain_google_genai import GoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain.chains import LLMChain

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

## 1. Displacements

The [Gaza Maps](https://gazamaps.com/) project maintains a database of all known IDF evacuation orders issued on official IDF Arabic channels, or via leaflets in Gaza. 

In [41]:
# Load the displacement data
displacement = pd.read_csv("data/displacement_updated.csv")
displacement.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 109 entries, 0 to 108
Data columns (total 21 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   date                     109 non-null    object 
 1   x_source                 93 non-null     object 
 2   x_text                   93 non-null     object 
 3   x_text_translated        93 non-null     object 
 4   x_timestamp_utc3         93 non-null     object 
 5   evacuation_zone_prompt   109 non-null    object 
 6   facebook_source          21 non-null     object 
 7   facebook_timestamp_utc3  21 non-null     object 
 8   leaflet                  19 non-null     object 
 9   leaflet_text_translated  18 non-null     object 
 10  safezone_prompt          109 non-null    object 
 11  link                     95 non-null     object 
 12  map_idf                  95 non-null     object 
 13  map_full                 79 non-null     object 
 14  map_zoom                 7

### Data cleaning
1. Remove the URL at the end of every tweet in the column `x_text`.
2. Translate every tweet from Arabic to English, creating a new column `x_text_translated`.
3. Convert the column `x_timestamp` to UTC+3, which is the timezone in Gaza.

In [42]:
def remove_urls(text):
    """Remove all URLs from the text."""
    if pd.isna(text) or not isinstance(text, str):
        return text # Leave non-string types unchanged
    return re.sub(r'https://\S+', '', text).strip()
    

def arabic_to_english(text):
    """Translate Arabic text to English."""
    if pd.isna(text) or not isinstance(text, str):
        return text
    try:
        return GoogleTranslator(source='ar', target='en').translate(text)
    except Exception as e:
        print(f"Error translating text: {text} - {e}")
        return text


def convert_date(time_str):
    """Convert a Twitter-style timestamp to hour:minute in UTC+3."""
    if pd.isna(time_str) or not isinstance(time_str, str):
        return time_str
    try:
        # Parse the original UTC datetime string
        dt = datetime.strptime(time_str, "%a %b %d %H:%M:%S %z %Y")     
        # Add 3 hours to get UTC+3
        dt_utc3 = dt + timedelta(hours=3)       
        # Format hour and minute
        time_formatted = dt_utc3.strftime("%H:%M")
        return time_formatted
    except Exception as e:
        print(f"Skipping malformed timestamp '{time_str}': {e}")
        return time_str

In [43]:
# Remove URLs from the tweet text
displacement['x_text'] = displacement['x_text'].apply(remove_urls)

# Translate Arabic tweets to English
# displacement['x_text_translated'] = displacement['x_text'].apply(arabic_to_english)

# Convert the timestamp to UTC+3 hour:minute format
# displacement['x_timestamp_utc3'] = displacement['x_timestamp'].apply(convert_date)

# Remove useless columns
displacement = displacement.drop(['evacuation_zone_prompt', 'safezone_prompt', 'map_full', 'map_zoom'], axis=1)

displacement.head()

,date,x_source,x_text,x_text_translated,x_timestamp_utc3,facebook_source,facebook_timestamp_utc3,leaflet,leaflet_text_translated,link,map_idf,displacement_blocks,labeled_safe_blocks,evacuation_zone,safezone,area_sq_km_displacement,area_sq_km_labeled_safe
0,2025-05-26,https://x.com/AvichayAdraee/status/19269564320...,#عاجل ‼️ الى سكان محافظة خانيونس، بني سهيلا، ع...,"To the residents of Khan Yunis, Bani Suhaila, ...",13:59,https://www.facebook.com/IDFarabicAvichayAdrae...,14:03,NaN,NaN,https://gazamaps.com/displacement/103,https://gazamaps.com/storage/displacement-maps...,"1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14,...",NaN,"Khan Yunis, Bani Suhaila, Abasan, Al-Qarara Go...",Al-Mawasi,154.44,0.0
1,2025-05-21,https://x.com/AvichayAdraee/status/19252451965...,#عاجل ‼️ تحذير خطير الى كل سكان قطاع غزة المتو...,A serious warning to all residents of the Gaza...,20:39,https://www.facebook.com/IDFarabicAvichayAdrae...,20:41,NaN,NaN,https://gazamaps.com/displacement/101,https://gazamaps.com/storage/displacement-maps...,"961, 962, 963, 964, 965, 966, 967, 968, 969, 9...",NaN,"Gaza Strip, Ghaban, Al-Shimaa, Fadous, Al-Mans...",south,13.26,0.0
2,2025-05-19,https://x.com/AvichayAdraee/status/19243953249...,#عاجل ‼️ الى سكان محافظة خان يونس، بني سهيلا و...,"To the residents of Khan Yunis, Bani Suhaila a...",12:22,https://www.facebook.com/IDFarabicAvichayAdrae...,12:23,NaN,NaN,https://gazamaps.com/displacement/100,https://gazamaps.com/storage/displacement-maps...,"36, 37.1, 37.2, 38, 39, 40, 41, 42.1, 42.2, 42...",NaN,"Khan Yunis, Bani Suhaila, Abasan",Mawasi area,81.24,0.0
3,2025-05-18,https://x.com/avichayadraee/status/19241385814...,#عاجل ‼️ إلى جميع سكان قطاع غزة المتواجدين في...,To all residents of the Gaza Strip located in ...,19:22,https://www.facebook.com/IDFarabicAvichayAdrae...,19:23,NaN,NaN,https://gazamaps.com/displacement/99,https://gazamaps.com/storage/displacement-maps...,"126, 127, 129.1, 129.2, 130.1, 130.2, 131.1, 2...",NaN,"Gaza Strip, Al-Qarara, Salqa, southern Deir al...",Al Mawasi,5.71,0.0
4,2025-05-16,NaN,NaN,NaN,NaN,NaN,NaN,https://idfleaflets.com/53,URGENT WARNING:\n\nTo all those who are in thi...,NaN,NaN,NaN,NaN,Unknown,Unknown,NaN,NaN


The tweet text `x_text_translated` contains the names of the locations designated to be evacuated, and the locations where residents should find shelter. We will use an LLM to extract these locations and put them in separate columns for further analysis.

> Note: Method currently not working. We used the SheetGPT extension for Google Sheets instead with the same prompts.

In [ ]:
try:
    # Load Google Gemini API key
    dotenv_path = os.path.abspath(os.path.join(os.getcwd(), '..', '.env'))
    load_dotenv(dotenv_path)

    google_api_key = os.getenv("GOOGLE_API_KEY")
except:
    google_api_key = input("Your Google API key: ")

# Set up the LLM
llm = GoogleGenerativeAI(model="gemini-2.0-flash", google_api_key=google_api_key)

# Global delay tracker
last_call_time = 0

async def throttled_llm_run(chain, input_data, delay=4):
    """Run LLM chain with rate limit delay (default: 4 seconds)."""
    global last_call_time

    now = time.time()
    time_since_last = now - last_call_time

    if time_since_last < delay:
        await asyncio.sleep(delay - time_since_last)
    
    try:
        result = chain.run(x_text_translated=input_data).strip()
        last_call_time = time.time()  # Update the last call time
        return result
    except Exception as e:
        print(f"Error running LLM chain: {e}")
        return None

In [ ]:
async def extract_evacuations(row):
    """Function to extract the names of locations designated to be evacuated from a tweet."""
    prompt = PromptTemplate.from_template("""
    You will be given a tweet containing an evacuation order to people in specific locations.
    Name the affected locations (not the block numbers, if present in the tweet) where people are ordered to evacuate from, separated by commas.
    If the location names where people should evacuate from are not mentioned in the tweet, answer "Unknown".
    Do not include the area which people are ordered to evacuate to.
                                          
    Tweet: {x_text_translated}
    """)

    chain = LLMChain(llm=llm, prompt=prompt)

    if isinstance(row['x_text_translated'], str):
        # Prompt the LLM to extract evacuation locations
        return await throttled_llm_run(chain, row['x_text_translated'])
    return None


async def extract_safezones(row):
    """Function to extract the names of safe zones from a tweet."""
    prompt = PromptTemplate.from_template("""
    You will be given a tweet containing an evacuation order to people in specific locations.
    Name the safe zones (not the block numbers, if present in the tweet) where people are ordered to evacuate to, separated by commas.
    If the safe zones where people should evacuate to are not mentioned in the tweet, answer "Unknown".
    Do not include the area which people are ordered to evacuate from.
                                          
    Tweet: {x_text_translated}
    """)

    chain = LLMChain(llm=llm, prompt=prompt)

    if isinstance(row['x_text_translated'], str):
        # Prompt the LLM to extract safe zone locations
        return await throttled_llm_run(chain, row['x_text_translated'])
    return None

In [ ]:
# Extract evacuation locations from the translated tweets
# displacement["locations"] = displacement.apply(extract_evacuations, axis=1)

# Extract safe zones from the translated tweets
# displacement["safezones"] = displacement.apply(extract_safezones, axis=1)

## 2. Political violence events

ACLED's [Gaza Monitor](https://acleddata.com/gaza-monitor/#1738833681742-00591c0f-83dc) contains a map and accompanying dataset of all political violence events, demonstration events, and strategic developments recorded in Israel and Palestine started from 7 October 2023.

In [44]:
# Load the ACLED data
acled = pd.read_csv("data/acled_may23.csv")
acled.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 90242 entries, 0 to 90241
Data columns (total 31 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   event_id_cnty       90242 non-null  object 
 1   event_date          90242 non-null  object 
 2   year                90242 non-null  int64  
 3   time_precision      90242 non-null  int64  
 4   disorder_type       90242 non-null  object 
 5   event_type          90242 non-null  object 
 6   sub_event_type      90242 non-null  object 
 7   actor1              90242 non-null  object 
 8   assoc_actor_1       20743 non-null  object 
 9   inter1              90242 non-null  object 
 10  actor2              66166 non-null  object 
 11  assoc_actor_2       16778 non-null  object 
 12  inter2              66166 non-null  object 
 13  interaction         90242 non-null  object 
 14  civilian_targeting  21230 non-null  object 
 15  iso                 90242 non-null  int64  
 16  regi

### Data cleaning

The [ACLED Codebook](https://acleddata.com/acleddatanew/wp-content/uploads/dlm_uploads/2024/10/ACLED-Codebook-2024-7-Oct.-2024.pdf) contains more information about how ACLED codes and catagorizes data. Based on this we will filter the dataset to only include battles, explosions/remote violence and violence against civilians by the IDF in Gaza after 7 October 2023:

1. Only include dates in the column `event_date` after 2023-10-07.
2. Only include datapoints for 'Political violence' in the column `disorder_type`. Only include the following subcategories within the column `event_type`:
    - Battles
    - Explosions/Remote violence
    - Violence against civilians
3. Only include datapoints that contain 'Military Forces of Israel' in the column `actor1`.
4. Only include datapoints for the location 'Gaza Strip' in the column `admin1`.

In [45]:
# Ensure event_date is in datetime format
acled["event_date"] = pd.to_datetime(acled["event_date"], errors='coerce')

# Filter events types
event_types = ["Battles", "Explosions/Remote violence", "Violence against civilians"]

# Apply filters
acled_filtered = acled[
    (acled["event_date"] > pd.Timestamp("2023-10-07")) &
    (acled["disorder_type"] == "Political violence") &
    (acled["event_type"].isin(event_types)) &
    (acled["actor1"].str.contains("Military Forces of Israel", na=False)) &
    (acled["admin1"] == "Gaza Strip")
]

acled_filtered.head()

,event_id_cnty,event_date,year,time_precision,disorder_type,event_type,sub_event_type,actor1,assoc_actor_1,inter1,...,location,latitude,longitude,geo_precision,source,source_scale,notes,fatalities,tags,timestamp
4,PSE74167,2025-05-23,2025,1,Political violence,Explosions/Remote violence,Air/drone strike,Military Forces of Israel (2022-),NaN,External/Other forces,...,Jabalya,31.5272,34.4835,2,Newpress; Palestine News and Information Agenc...,New media-National,"On 23 May 2025, Israeli warplanes targeted a g...",3,NaN,1748304023
5,PSE74182,2025-05-23,2025,1,Political violence,Explosions/Remote violence,Air/drone strike,Military Forces of Israel (2022-),NaN,External/Other forces,...,Gaza - Southern Remal,31.5168,34.4355,2,Quds News Network,National,"On 23 May 2025, Israeli warplanes targeted sev...",0,NaN,1748304023
6,PSE74194,2025-05-23,2025,1,Political violence,Explosions/Remote violence,Air/drone strike,Military Forces of Israel (2022-),NaN,External/Other forces,...,An Nusayrat,31.4486,34.3925,1,Palestine News and Information Agency; Quds Ne...,National,"On 23 May 2025, Israeli warplanes targeted the...",3,NaN,1748304023
7,PSE74204,2025-05-23,2025,1,Political violence,Explosions/Remote violence,Air/drone strike,Military Forces of Israel (2022-),NaN,External/Other forces,...,Abasan al Jadidah,31.3416,34.3459,1,Quds News Network,National,"On 23 May 2025, Israeli warplanes targeted a h...",14,NaN,1748304023
8,PSE74213,2025-05-23,2025,1,Political violence,Explosions/Remote violence,Air/drone strike,Military Forces of Israel (2022-),NaN,External/Other forces,...,Al Qararah,31.3739,34.3409,2,Palestine News and Information Agency; Quds Ne...,National,"On 23 May 2025, Israeli helicopters fired live...",1,NaN,1748304024


## 3. Merging the data